# Faruq-v3 grouped development split

Notebook ini menghapus leakage `source_parent_id` antara train dan validation pada Faruq-v2 yang geometrinya sudah diperbaiki. Ia tidak training, inference, atau membuka test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
command = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(command)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)


In [ ]:
import tarfile
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

PROJECT_ROOT = resolve_drive_project_root()
V2_ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v2.tar')
V2_ROOT = Path('/content/faruq-development-v2')
V3_ROOT = Path('/content/faruq-development-v3-grouped')
EVIDENCE_ROOT = PROJECT_ROOT / 'evidence/faruq-grouped-development-v1'
V3_ARCHIVE = PROJECT_ROOT / 'bundles/faruq-development-v3-grouped.tar'
assert V2_ARCHIVE.is_file(), f'Arsip Faruq-v2 tidak ditemukan: {V2_ARCHIVE}'
if not (V2_ROOT / 'faruq_geometry_repair_manifest.json').is_file():
    with tarfile.open(V2_ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (V2_ROOT / 'faruq_geometry_repair_manifest.json').is_file(), V2_ROOT
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=True)
print('FARUQ V2:', V2_ROOT)
print('FARUQ V3:', V3_ROOT)


In [ ]:
import json
from coffee_detector.group_faruq_development import group_faruq_development

summary = group_faruq_development(V2_ROOT, V3_ROOT, seed=42, val_fraction=0.15, link_mode='auto')
assert summary['training_executed'] is False
assert summary['inference_executed'] is False
assert summary['test_images_accessed'] is False
assert summary['cross_split_parent_identities'] == 0
assert summary['cross_split_exact_hashes'] == 0
print(json.dumps(summary, indent=2, ensure_ascii=False))
for name in ('faruq_grouped_summary.json', 'faruq_grouped_manifest.json', 'faruq_group_manifest.json', 'dataset_audit.json'):
    shutil.copy2(V3_ROOT / name, EVIDENCE_ROOT / name)


In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for class_name in sorted(summary['annotations_by_split_and_class']['train']):
    rows.append({
        'class_name': class_name,
        'train': summary['annotations_by_split_and_class']['train'][class_name],
        'val': summary['annotations_by_split_and_class']['val'][class_name],
    })
display(pd.DataFrame(rows))
print('GATES:', json.dumps(summary['gates'], indent=2))
print('TRAINING READY:', summary['training_ready'])


In [ ]:
if summary['training_ready'] and not V3_ARCHIVE.is_file():
    temporary = Path('/content/faruq-development-v3-grouped.tar')
    with tarfile.open(temporary, 'w') as archive:
        archive.add(V3_ROOT, arcname='faruq-development-v3-grouped')
    shutil.copy2(temporary, V3_ARCHIVE)
    temporary.unlink()
if V3_ARCHIVE.is_file():
    print('ARCHIVE:', V3_ARCHIVE, f'({V3_ARCHIVE.stat().st_size / 1024**2:.1f} MB)')
else:
    print('ARCHIVE TIDAK DIBUAT karena gate belum PASS.')
print('SUMMARY:', EVIDENCE_ROOT / 'faruq_grouped_summary.json')
print('Kirim tabel kelas dan seluruh gates. Jangan training dahulu.')
